# Phase 06B.03C — QTG/TATG selector training

Actual two-stage train-side loop. QTG and TATG use independent initialization/output directories; no Vintern, adapter or encoder parameter enters either optimizer.

In [ ]:
from pathlib import Path
import json,sys,gc
import torch
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/"src").is_dir()),None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))
from roadbuddy_common import save_json
from phase06a_common import sha256_file
from phase06b_common import assert_selected_track,validate_feature_bank_manifest
from traffic_temporal_grounding import TemporalGroundingConfig,TrafficAwareTemporalGrounder,run_selector_training_stage
RUN_SCOPE="smoke" # smoke | full
PROTOCOL=ROOT/"outputs/phase06b/novelty_protocol/novelty_protocol.json"; BANKS=ROOT/"outputs/phase06b/feature_banks"/RUN_SCOPE
OUT=ROOT/"outputs/phase06b/selector_training"/RUN_SCOPE; OUT.mkdir(parents=True,exist_ok=True); CONFIG=ROOT/"outputs/phase06b/selector_training/selector_training_config.json"
if not PROTOCOL.is_file(): raise RuntimeError("Human novelty decision is not locked")
protocol=json.loads(PROTOCOL.read_text()); assert_selected_track(protocol,"traffic_temporal_grounding")


In [ ]:
if not CONFIG.is_file():
    save_json(ROOT/"outputs/phase06b/selector_training/selector_training_config.template.json",{"status":"train_side_locked","seed":42,"hidden_dim":256,"dropout":0.1,"min_temporal_gap":0.0,"learning_rate":0.0001,"weight_decay":0.01,"max_grad_norm":1.0,"gradient_accumulation":16,"stage_a_epochs":20,"stage_b_epochs":20,"evaluation_steps":8,"patience_evaluations":3,"primary_metric":"inner_dev_grounding_loss"})
    save_json(OUT/"PHASE06B_03C_STATUS.json",{"status":"awaiting_train_side_hyperparameter_lock","scope":RUN_SCOPE,"scientific_complete":False})
    raise RuntimeError("Lock selector hyperparameters using train-side development only")
cfg=json.loads(CONFIG.read_text()); required=["seed","hidden_dim","dropout","min_temporal_gap","learning_rate","weight_decay","max_grad_norm","gradient_accumulation","stage_a_epochs","stage_b_epochs","evaluation_steps","patience_evaluations"]
if cfg.get("status")!="train_side_locked" or any(k not in cfg for k in required): raise ValueError("Selector config is not train-side locked")
manifests={}
for split in ["train_fit","inner_dev"]:
    mp=BANKS/split/"feature_bank_manifest.json"; bp=BANKS/split/"feature_bank.pt"
    manifests[split]=validate_feature_bank_manifest(json.loads(mp.read_text()))
    if not bp.is_file() or sha256_file(bp)!=manifests[split]["bank_sha256"]: raise ValueError(f"{split} feature bank hash mismatch")
fit=torch.load(BANKS/"train_fit/feature_bank.pt",map_location="cpu",weights_only=False); dev=torch.load(BANKS/"inner_dev/feature_bank.pt",map_location="cpu",weights_only=False)
if {str(r["group_id"]) for r in fit}&{str(r["group_id"]) for r in dev}: raise ValueError("train_fit/inner_dev group leakage")


In [ ]:
results={}
for family in ["QTG","TATG"]:
    use_traffic=family=="TATG"; family_fit=[dict(r) for r in fit]; family_dev=[dict(r) for r in dev]
    if not use_traffic:
        for r in [*family_fit,*family_dev]: r.pop("traffic_features",None)
    traffic_dim=int(manifests["train_fit"].get("traffic_feature_dim",0)) if use_traffic else 0
    if use_traffic and traffic_dim<=0:
        save_json(OUT/"PHASE06B_03C_STATUS.json",{"status":"awaiting_traffic_features","qtag_train_side_ready":True,"scope":RUN_SCOPE,"scientific_complete":False}); raise RuntimeError("TATG requires the locked traffic feature extractor/bank")
    model_cfg=TemporalGroundingConfig(frame_feature_dim=int(manifests["train_fit"]["frame_feature_dim"]),question_feature_dim=int(manifests["train_fit"]["question_feature_dim"]),traffic_feature_dim=traffic_dim,hidden_dim=int(cfg["hidden_dim"]),dropout=float(cfg["dropout"]),candidate_count=32,selected_count=3,min_temporal_gap=float(cfg["min_temporal_gap"]))
    torch.manual_seed(int(cfg["seed"])); stage_a_model=TrafficAwareTemporalGrounder(model_cfg)
    stage_a=run_selector_training_stage(stage_a_model,family_fit,output_dir=OUT/family/"stage_a_inner_selection",epochs=int(cfg["stage_a_epochs"]),gradient_accumulation=int(cfg["gradient_accumulation"]),learning_rate=float(cfg["learning_rate"]),weight_decay=float(cfg["weight_decay"]),max_grad_norm=float(cfg["max_grad_norm"]),evaluation_records=family_dev,evaluation_steps=int(cfg["evaluation_steps"]),patience_evaluations=int(cfg["patience_evaluations"]),seed=int(cfg["seed"]))
    locked_step=int(stage_a["best_step"]); del stage_a_model; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    torch.manual_seed(int(cfg["seed"])); stage_b_model=TrafficAwareTemporalGrounder(model_cfg)
    stage_b=run_selector_training_stage(stage_b_model,[*family_fit,*family_dev],output_dir=OUT/family/"stage_b_final_retrain",epochs=int(cfg["stage_b_epochs"]),gradient_accumulation=int(cfg["gradient_accumulation"]),learning_rate=float(cfg["learning_rate"]),weight_decay=float(cfg["weight_decay"]),max_grad_norm=float(cfg["max_grad_norm"]),max_optimizer_steps=locked_step,evaluation_records=None,seed=int(cfg["seed"]))
    if stage_b["optimizer_steps"]!=locked_step: raise RuntimeError("Stage B did not reach the locked optimizer step")
    results[family]={"locked_optimizer_step":locked_step,"checkpoint_path":stage_b["final_checkpoint"],"checkpoint_sha256":sha256_file(Path(stage_b["final_checkpoint"])),"trainable_parameters":stage_b["trainable_parameters"],"runtime_seconds":stage_a["runtime_seconds"]+stage_b["runtime_seconds"],"peak_vram_bytes":max(stage_a["peak_vram_bytes"],stage_b["peak_vram_bytes"])}
    del stage_b_model; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
save_json(OUT/"selector_training_summary.json",results); save_json(OUT/"PHASE06B_03C_STATUS.json",{"status":"complete" if RUN_SCOPE=="full" else "smoke_complete","scope":RUN_SCOPE,"scientific_complete":False,"families":results})
results
